In [23]:
import pandas as pd

In [24]:
train_data = pd.read_csv("../datasets/samsum-train.csv")
val_data = pd.read_csv("../datasets/samsum-validation.csv")

In [25]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [26]:
train_data.isna().sum()

id          0
dialogue    1
summary     0
dtype: int64

In [27]:
train_data.shape

(14732, 3)

In [28]:
train_data.dropna(inplace=True)
val_data.dropna(inplace=True)

In [29]:
train_data.isna().sum()

id          0
dialogue    0
summary     0
dtype: int64

In [30]:
train_data.shape

(14731, 3)

In [31]:
import re

In [32]:
def clean_data(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip().lower()
    return text

In [33]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)
val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [34]:
from transformers import T5Tokenizer

In [35]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [36]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"], max_length=512, padding="max_length", truncation=True)
    targets = tokenizer(data["summary"], max_length=150, padding="max_length", truncation=True)
    inputs["labels"] = targets["input_ids"]
    return inputs

train_data = train_data.apply(tokenize, axis=1).tolist()
val_data = val_data.apply(tokenize, axis=1).tolist()

In [37]:
from transformers import T5ForConditionalGeneration

In [38]:
model = T5ForConditionalGeneration.from_pretrained("t5-small", device_map="auto")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [39]:
# Device check
import torch
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [40]:
print(next(model.parameters()).device)

mps:0


In [41]:
from transformers import TrainingArguments, Trainer

In [42]:
training_args = TrainingArguments(
    output_dir="../model_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=300,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    dataloader_pin_memory=False
)


In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.367484,0.328022
2,0.345894,0.316231


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
save_path = "../models"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model and tokenizer saved to {save_path}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model and tokenizer saved successfully to ../models!
